** Importing Libraries**

In [ ]:
import argparse
import math
import os
import random
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

!pip install opacus
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator

**Load Dataset**

In [ ]:
from collections import Counter

print("\n" + "="*70)
print("Loading Dataset")
print("="*70)

url = "https://raw.githubusercontent.com/chuyq/MESC/main/MESC.csv"
df = pd.read_csv(url)

TEXT_COL = "Utterance"
LABEL_COL = "Emotion"

# Clean the dataset
df = df[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)
df = df[df[TEXT_COL].astype(str).str.strip().ne("")].reset_index(drop=True)

print(f"Dataset loaded successfully. Total samples: {len(df)}")
print(f"\nClass distribution:")
class_counts = Counter(df[LABEL_COL])
for emotion, count in sorted(class_counts.items()):
    print(f"  {emotion}: {count} ({count/len(df)*100:.1f}%)")


Loading Dataset
Dataset loaded successfully. Total samples: 28762

Class distribution:
  anger: 2659 (9.2%)
  depression: 4958 (17.2%)
  disgust: 1669 (5.8%)
  fear: 251 (0.9%)
  joy: 1350 (4.7%)
  neutral: 17116 (59.5%)
  sadness: 759 (2.6%)


**Patch torch.load**

In [ ]:
import torch.serialization as ts

_original_load = ts.load


def _patched_load(*args, **kwargs):

    try:
        kwargs.setdefault("weights_only", False)
        return _original_load(*args, **kwargs)
    except TypeError:

        kwargs.pop("weights_only", None)
        return _original_load(*args, **kwargs)


ts.load = _patched_load
torch.load = _patched_load

print("Patched torch.load / torch.serialization.load")

Patched torch.load / torch.serialization.load


**Utility Functions and Constants**

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


TEXT_COL = "Utterance"
LABEL_COL = "Emotion"

**MESC Dataset Class**

In [ ]:
class MESCDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_len: int = 128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding=False,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

Training and Evaluation Functions

In [ ]:

def train_one_epoch(model, dataloader, optimizer, scheduler, device, dp: bool, privacy_engine=None, delta=None, epoch_idx=None, num_epochs=None):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0.0

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"]
        outputs = model(**{k: v for k, v in batch.items() if k != "labels"})
        logits = outputs.logits

        loss = loss_fn(logits, labels)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / max(1, len(dataloader))

    if dp and privacy_engine is not None and delta is not None:
        eps = privacy_engine.get_epsilon(delta)
        print(f"  -> Train loss={avg_loss:.4f}, ε≈{eps:.2f} (epoch {epoch_idx}/{num_epochs})")
    else:
        print(f"  -> Train loss={avg_loss:.4f} (epoch {epoch_idx}/{num_epochs})")

    return avg_loss


def eval_model(model, dataloader, device, id2label=None):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            labels = batch["labels"].numpy()
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**{k: v for k, v in batch.items() if k != "labels"}).logits
            preds = logits.argmax(dim=-1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels)

    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")

    if id2label is not None:
        pred_str = [id2label[i] for i in all_preds]
        true_str = [id2label[i] for i in all_labels]
        print("\nClassification report:\n")
        print(classification_report(true_str, pred_str))
    else:
        print("\nClassification report (numeric labels):\n")
        print(classification_report(all_labels, all_preds))

    print(f"Eval accuracy={acc:.4f}, macro F1={f1_macro:.4f}")
    return acc, f1_macro

**Configuration Arguments**

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="Fine-tune DistilBERT on MESC with optional DP")

    parser.add_argument("--model_name", type=str, default="distilbert-base-uncased", help="HF model name")
    parser.add_argument("--max_len", type=int, default=128)
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--lr", type=float, default=2e-5)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--dp", action="store_true", help="Enable Differential Privacy with Opacus")
    parser.add_argument("--epsilon", type=float, default=8.0, help="Target epsilon for DP")
    parser.add_argument("--max_grad_norm", type=float, default=1.0, help="Gradient clipping norm for DP")
    parser.add_argument("--secure_mode", action="store_true", help="Opacus secure mode")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--test_size", type=float, default=0.2)
    parser.add_argument("--val_size", type=float, default=0.1, help="Fraction of TRAIN to use as validation")
    parser.add_argument("--subset", type=int, default=0, help="Optional: use only first N samples (debug)")


    return parser.parse_args(args=[])


**Main Training and Evaluation Loop**

In [ ]:
def main():
    args = parse_args()
    set_seed(args.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")


    print("Using globally available 'df' for data.")

    if TEXT_COL not in df.columns or LABEL_COL not in df.columns:
        raise ValueError(f"Expected columns '{TEXT_COL}' and '{LABEL_COL}' in CSV, found: {df.columns.tolist()}")

    texts = df[TEXT_COL].astype(str).tolist()
    labels_str = df[LABEL_COL].astype(str).tolist()

    if args.subset > 0:
        texts = texts[: args.subset]
        labels_str = labels_str[: args.subset]
        print(f"Using subset of size {len(texts)}")

    le = LabelEncoder()
    labels = le.fit_transform(labels_str)
    num_labels = len(le.classes_)
    id2label = {i: lab for i, lab in enumerate(le.classes_)}
    label2id = {lab: i for i, lab in id2label.items()}

    print(f"Classes ({num_labels}): {le.classes_}")

    X_train, X_test, y_train, y_test = train_test_split(
        texts,
        labels,
        test_size=args.test_size,
        random_state=args.seed,
        stratify=labels,
    )

    if args.val_size > 0:
        X_train, X_val, y_train, y_val = train_test_split(
            X_train,
            y_train,
            test_size=args.val_size,
            random_state=args.seed,
            stratify=y_train,
        )
        use_val = True
        print(f"Train size={len(X_train)}, val size={len(X_val)}, test size={len(X_test)}")
    else:
        X_val, y_val = [], []
        use_val = False
        print(f"Train size={len(X_train)}, test size={len(X_test)}")

    print(f"Loading tokenizer: {args.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, use_fast=True)

    train_dataset = MESCDataset(X_train, y_train, tokenizer, max_len=args.max_len)
    test_dataset = MESCDataset(X_test, y_test, tokenizer, max_len=args.max_len)
    val_dataset = MESCDataset(X_val, y_val, tokenizer, max_len=args.max_len) if use_val else None

    collator = DataCollatorWithPadding(tokenizer, padding="longest", return_tensors="pt")

    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        collate_fn=collator,
        drop_last=True if args.dp else False,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        collate_fn=collator,
    )
    val_loader = None
    if use_val:
        val_loader = DataLoader(
            val_dataset,
            batch_size=args.batch_size,
            shuffle=False,
            collate_fn=collator,
        )

    print(f"Loading model: {args.model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(
        args.model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    ).to(device)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=args.lr,
        weight_decay=args.weight_decay,
    )

    total_steps = args.epochs * max(1, len(train_loader))
    warmup_steps = int(0.06 * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    dp_enabled = bool(args.dp)
    privacy_engine = None
    delta = None

    if dp_enabled:
        print("Enabling Differential Privacy with Opacus")

        if not ModuleValidator.is_valid(model):
            print("Model is not valid for DP, applying ModuleValidator.fix...")
            model = ModuleValidator.fix(model)

        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=args.lr,
            weight_decay=args.weight_decay,
        )

        n_train = len(train_dataset)
        delta = 1.0 / max(1, n_train)
        print(f"Using delta={delta:.2e} (1/N), target epsilon={args.epsilon}")

        privacy_engine = PrivacyEngine(secure_mode=args.secure_mode)

        model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
            module=model,
            optimizer=optimizer,
            data_loader=train_loader,
            epochs=args.epochs,
            target_epsilon=args.epsilon,
            target_delta=delta,
            max_grad_norm=args.max_grad_norm,
        )

    best_val_f1 = -1.0
    best_state_dict = None

    for epoch in range(1, args.epochs + 1):
        print(f"\n===== Epoch {epoch}/{args.epochs} =====")
        train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            device,
            dp=dp_enabled,
            privacy_engine=privacy_engine,
            delta=delta,
            epoch_idx=epoch,
            num_epochs=args.epochs,
        )

        if use_val:
            print("Evaluating on validation set...")
            _, val_f1 = eval_model(model, val_loader, device, id2label=id2label)
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                print(f"  -> New best val macro F1={best_val_f1:.4f}, saving state dict")
        else:
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}


    if best_state_dict is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state_dict.items()})

    print("\n===== Final evaluation on TEST set =====")
    acc, f1_macro = eval_model(model, test_loader, device, id2label=id2label)

    if dp_enabled and privacy_engine is not None and delta is not None:
        final_eps = privacy_engine.get_epsilon(delta)
        print(f"\nFinal spent epsilon \u2248 {final_eps:.2f}, delta={delta:.2e}")
    else:
        final_eps = None

    out_dir = "mesc_dp_model" if dp_enabled else "mesc_nodp_model"
    os.makedirs(out_dir, exist_ok=True)
    print(f"\nSaving model and tokenizer to {out_dir}/")
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)

    metrics_path = os.path.join(out_dir, "metrics.txt")
    with open(metrics_path, "w") as f:
        f.write(f"accuracy={acc:.4f}\n")
        f.write(f"f1_macro={f1_macro:.4f}\n")
        if final_eps is not None:
            f.write(f"epsilon={final_eps:.4f}\n")
            f.write(f"delta={delta:.6e}\n")

    print(f"Metrics written to {metrics_path}")
    print("Done.")


if __name__ == "__main__":
    main()

Using device: cuda
Using globally available 'df' for data.
Classes (7): ['anger' 'depression' 'disgust' 'fear' 'joy' 'neutral' 'sadness']
Train size=20708, val size=2301, test size=5753
Loading tokenizer: distilbert-base-uncased


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

  -> New best val macro F1=0.1916, saving state dict

===== Epoch 3/3 =====
  -> Train loss=0.9492 (epoch 3/3)
Evaluating on validation set...

Classification report:

              precision    recall  f1-score   support

       anger       0.35      0.24      0.29       213
  depression       0.37      0.38      0.38       397
     disgust       0.22      0.03      0.05       133
        fear       0.00      0.00      0.00        20
         joy       0.25      0.10      0.14       108
     neutral       0.69      0.85      0.76      1369
     sadness       0.00      0.00      0.00        61

    accuracy                           0.60      2301
   macro avg       0.27      0.23      0.23      2301
weighted avg       0.53      0.60      0.55      2301

Eval accuracy=0.5976, macro F1=0.2313


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


  -> New best val macro F1=0.2313, saving state dict

===== Final evaluation on TEST set =====

Classification report:

              precision    recall  f1-score   support

       anger       0.27      0.18      0.21       532
  depression       0.32      0.33      0.33       992
     disgust       0.17      0.02      0.03       334
        fear       0.00      0.00      0.00        50
         joy       0.25      0.09      0.14       270
     neutral       0.68      0.85      0.76      3423
     sadness       0.00      0.00      0.00       152

    accuracy                           0.58      5753
   macro avg       0.24      0.21      0.21      5753
weighted avg       0.51      0.58      0.53      5753

Eval accuracy=0.5825, macro F1=0.2093

Saving model and tokenizer to mesc_nodp_model/


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Metrics written to mesc_nodp_model/metrics.txt
Done.
